# Phase 4C.1: Colab T4 Training Launcher

Launcher thực thi ma trận Stage 1 (15 runs) trên NVIDIA T4.
Upload 3 file vào `/content/forensics-transfer/`:
1. `phase_4c1_code_79bb115.tar.gz`
2. `phase_4c1_binary_n250_reusable.tar`
3. `phase_4c1_t4_execute_all_stage1.sh`

In [ ]:
# Cell 1: Configuration
from pathlib import Path

TRANSFER_DIR = Path("/content/forensics-transfer")

EXPECTED_ARTIFACTS = {
    "code_archive": {
        "filename": "phase_4c1_code_79bb115.tar.gz",
        "bytes": 758350,
        "sha256": "5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5"
    },
    "reusable_bundle": {
        "filename": "phase_4c1_binary_n250_reusable.tar",
        "bytes": 724633600,
        "sha256": "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
    },
    "operator_script": {
        "filename": "phase_4c1_t4_execute_all_stage1.sh",
        "bytes": 20005,
        "sha256": "deb3f04dd7c1039c4f3248f98a27f4eaf8e11f206a218125e949e5869da8ce05"
    }
}

# Set EXECUTE = True to run training on Colab T4
EXECUTE = False
print(f"Configuration loaded. EXECUTE = {EXECUTE}")


In [ ]:
# Cell 2: Preflight
import sys
import hashlib
import torch

print(f"Platform: {sys.platform}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

gpu_model = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"GPU: {gpu_model}")

assert sys.platform.startswith("linux"), f"Linux required, got {sys.platform}"

if EXECUTE:
    assert torch.cuda.is_available(), "CUDA must be available when EXECUTE=True"
    assert "T4" in gpu_model.upper(), f"NVIDIA T4 required, got {gpu_model}"

assert TRANSFER_DIR.exists(), f"Transfer directory missing: {TRANSFER_DIR}"

def compute_sha256(file_path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest().lower()

for key, spec in EXPECTED_ARTIFACTS.items():
    fp = TRANSFER_DIR / spec["filename"]
    assert fp.exists(), f"Missing file: {fp}"
    act_bytes = fp.stat().st_size
    assert act_bytes == spec["bytes"], f"Byte mismatch for {spec['filename']}: expected {spec['bytes']}, got {act_bytes}"
    act_sha = compute_sha256(fp)
    assert act_sha == spec["sha256"].lower(), f"SHA-256 mismatch for {spec['filename']}: expected {spec['sha256']}, got {act_sha}"
    print(f"[+] Verified {spec['filename']}: {act_bytes} bytes, SHA-256 match")

print("[+] Preflight verification completed successfully.")


In [ ]:
# Cell 3: Execution
import subprocess

if not EXECUTE:
    print("[DRY-RUN] EXECUTE is False. Operator not invoked.")
    print("[DRY-RUN] Set EXECUTE = True in Cell 1 to execute the 15-run Stage 1 matrix.")
else:
    operator_cmd = ["bash", "/content/forensics-transfer/phase_4c1_t4_execute_all_stage1.sh"]
    print(f"[*] Launching operator: {' '.join(operator_cmd)}")
    subprocess.run(operator_cmd, check=True)
    print("[+] Operator finished successfully.")


In [ ]:
# Cell 4: Result handoff
from pathlib import Path

DOWNLOAD_DIR = Path("/content/phase_4c1_outputs/download")

REQUIRED_ARCHIVES = [
    "n50_results.tar.gz",
    "n100_results.tar.gz",
    "n250_results.tar.gz",
    "phase_4c1_t4_execution_logs.tar.gz",
    "phase_4c1_all_15_runs_results.tar.gz"
]

if not EXECUTE:
    print("[DRY-RUN] EXECUTE is False. Output check skipped.")
else:
    assert DOWNLOAD_DIR.exists(), f"Download directory missing: {DOWNLOAD_DIR}"
    print("Final output archives ready for download:")
    for name in REQUIRED_ARCHIVES:
        archive_path = DOWNLOAD_DIR / name
        assert archive_path.exists(), f"Archive missing: {archive_path}"
        sha_path = DOWNLOAD_DIR / f"{name}.sha256"
        assert sha_path.exists(), f"Checksum file missing: {sha_path}"
        bytes_count = archive_path.stat().st_size
        print(f"  - {archive_path} ({bytes_count:,} bytes)")
    print("[+] All output archives verified.")
